# Notebook A — Baseline

LoRA rank=16, q+v only, 3 epochs, train split only, IMG_SIZE=224.

In [ ]:
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow tqdm

In [ ]:
import os, json, random, ast
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoProcessor, AutoModelForVision2Seq,
    BitsAndBytesConfig, get_cosine_schedule_with_warmup,
)
from peft import get_peft_model, LoraConfig, TaskType, PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DATA_DIR    = Path("/kaggle/input/competitions/pixels-to-predictions")
OUTPUT_DIR  = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
ADAPTER_DIR = OUTPUT_DIR / "lora_adapter"
MODEL_ID    = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LABELS = "ABCDEFGH"
IMG_SIZE      = 224
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
LEARNING_RATE = 2e-4
WEIGHT_DECAY  = 0.01
NUM_EPOCHS    = 3
TRAIN_BATCH   = 4
GRAD_ACCUM    = 4
MAX_SEQ_LEN   = 1024
WARMUP_RATIO  = 0.05

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_4BIT = torch.cuda.is_available()
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
def load_split(split):
    df = pd.read_csv(DATA_DIR / f"{split}.csv")
    df["choices"] = df["choices"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    return df

train_df = load_split("train")
val_df   = load_split("val")
test_df  = load_split("test")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")


In [ ]:
train_img_dir = DATA_DIR / "images/images/train"
actual_images = set(os.listdir(train_img_dir))
train_df["img_filename"] = train_df["image_path"].apply(lambda x: x.split("/")[-1])
missing = train_df[~train_df["img_filename"].isin(actual_images)]
print(f"Images on disk: {len(actual_images)} | CSV rows: {len(train_df)} | Missing: {len(missing)}")


In [ ]:
def build_prompt(row, include_answer=False):
    parts = ["<image>"]
    lecture = row.get("lecture", None)
    if pd.notna(lecture) and str(lecture).strip():
        parts.append(f"Context:\n{str(lecture).strip()}")
    hint = row.get("hint", None)
    if pd.notna(hint) and str(hint).strip():
        parts.append(f"Hint:\n{str(hint).strip()}")
    parts.append(f"Question: {row['question'].strip()}")
    parts.append("Choices:")
    for i, c in enumerate(row["choices"]):
        parts.append(f"  {CHOICE_LABELS[i]}. {c}")
    parts.append("Answer:")
    if include_answer:
        parts.append(CHOICE_LABELS[int(row["answer"])])
    return "\n".join(parts)


In [ ]:
class ScienceQATrainDataset(Dataset):
    def __init__(self, df, data_dir, img_size=224):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
    def _load_image(self, rel_path):
        # Fix double-nested images directory
        fixed = str(rel_path).replace("images/", "images/images/", 1)
        return (Image.open(self.data_dir / fixed)
                .convert("RGB")
                .resize((self.img_size, self.img_size), Image.BICUBIC))
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {"image": self._load_image(row["image_path"]),
                "prompt": build_prompt(row, include_answer=True),
                "answer_idx": int(row["answer"])}

class ScienceQAEvalDataset(Dataset):
    def __init__(self, df, data_dir, img_size=224):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
    def _load_image(self, rel_path):
        # Fix double-nested images directory
        fixed = str(rel_path).replace("images/", "images/images/", 1)
        return (Image.open(self.data_dir / fixed)
                .convert("RGB")
                .resize((self.img_size, self.img_size), Image.BICUBIC))
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {"id": row["id"],
                "image": self._load_image(row["image_path"]),
                "prompt": build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer_idx": int(row["answer"]) if "answer" in row and pd.notna(row.get("answer")) else -1}

train_ds = ScienceQATrainDataset(train_df, DATA_DIR, img_size=IMG_SIZE)
val_ds   = ScienceQAEvalDataset(val_df,   DATA_DIR, img_size=IMG_SIZE)
test_ds  = ScienceQAEvalDataset(test_df,  DATA_DIR, img_size=IMG_SIZE)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
print(f"Base model loaded. Total params: {sum(p.numel() for p in base_model.parameters()):,}")


In [ ]:
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],
    bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
if torch.cuda.is_available():
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()


In [ ]:
def train_collate_fn(batch):
    images  = [item["image"]  for item in batch]
    prompts = [item["prompt"] for item in batch]
    encoding = processor(text=prompts, images=images, return_tensors="pt",
                         padding=True, truncation=True, max_length=MAX_SEQ_LEN)
    labels = encoding["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    encoding["labels"] = labels
    return encoding

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH, shuffle=True,
                          collate_fn=train_collate_fn, num_workers=2,
                          pin_memory=torch.cuda.is_available())
print(f"Training batches per epoch: {len(train_loader)}")


In [ ]:
optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps  = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"Total optimizer steps: {total_steps} | Warmup steps: {warmup_steps}")

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for step, batch in enumerate(pbar):
        batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        loss = model(**batch).loss / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        pbar.set_postfix(loss=f"{epoch_loss/(step+1):.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    print(f"Epoch {epoch+1} complete. Avg loss: {epoch_loss/len(train_loader):.4f}")

print("Training complete!")


In [ ]:
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to: {ADAPTER_DIR}")
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f} MB)")


In [ ]:
model.eval()
print('Model ready for inference.')

In [ ]:
@torch.inference_mode()
def score_choices_loglik(model, processor, image, base_prompt, choices):
    scores = []
    for i in range(len(choices)):
        full_prompt = base_prompt + " " + CHOICE_LABELS[i]
        inputs = processor(text=[full_prompt], images=[image],
                           return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
        labels = torch.full_like(inputs["input_ids"], -100)
        labels[:, -1] = inputs["input_ids"][:, -1]
        inputs["labels"] = labels
        outputs = model(**inputs)
        scores.append(-outputs.loss.item())
    return int(np.argmax(scores)), scores

print("Log-likelihood scoring function ready.")


In [ ]:
model.eval()
correct, total = 0, 0
val_results = []
for item in tqdm(val_ds, desc="Validating"):
    pred_idx, _ = score_choices_loglik(model, processor, item["image"], item["prompt"], item["choices"])
    gt_idx = item["answer_idx"]
    val_results.append({"id": item["id"], "pred": pred_idx, "gt": gt_idx, "correct": pred_idx == gt_idx})
    if pred_idx == gt_idx: correct += 1
    total += 1
    if total % 100 == 0:
        print(f"  [{total}/{len(val_ds)}] Running accuracy: {correct/total:.4f}")

print(f"\nFinal Validation Accuracy: {correct/total:.4f} ({correct}/{total})")
val_results_df = pd.DataFrame(val_results)
val_results_df.to_csv(OUTPUT_DIR / "val_results.csv", index=False)


In [ ]:
predictions = []
for i, item in enumerate(tqdm(test_ds, desc="Predicting on test")):
    pred_idx, _ = score_choices_loglik(model, processor, item["image"], item["prompt"], item["choices"])
    predictions.append({"id": item["id"], "answer": pred_idx})
    if (i+1) % 100 == 0:
        print(f"  [{i+1}/{len(test_ds)}] predictions generated")
print(f"\nGenerated {len(predictions)} predictions.")


In [ ]:
submission_df = pd.DataFrame(predictions)
assert list(submission_df.columns) == ["id", "answer"]
assert pd.api.types.is_integer_dtype(submission_df["answer"])
assert len(submission_df) == len(test_df)
assert set(submission_df["id"]) == set(test_df["id"])

submission_path = Path("/kaggle/working/submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Saved: {submission_path} | Rows: {len(submission_df)}")
print("\nAnswer distribution:")
print(submission_df["answer"].value_counts().sort_index())
print("\nFirst 5 rows:")
print(submission_df.head())
